In [1]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATASET_FOLDER = "/kaggle/input/uw-madison-gi-tract-image-segmentation"



In [2]:
df_train = pd.read_csv(os.path.join(DATASET_FOLDER, "train.csv"))
print(f"train size: {len(df_train)}")
display(df_train.head())



train size: 95088


,id,class,segmentation
0,case77_day20_slice_0001,large_bowel,NaN
1,case77_day20_slice_0001,small_bowel,NaN
2,case77_day20_slice_0001,stomach,NaN
3,case77_day20_slice_0002,large_bowel,NaN
4,case77_day20_slice_0002,small_bowel,NaN


In [3]:
df_ssub = pd.read_csv(os.path.join(DATASET_FOLDER, "sample_submission.csv"))
print(f"sample submission size: {len(df_ssub)}")
display(df_ssub.head())



sample submission size: 20400


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,1 1 5 2
1,case123_day20_slice_0001,small_bowel,1 1 5 2
2,case123_day20_slice_0001,stomach,1 1 5 2
3,case123_day20_slice_0002,large_bowel,1 1 5 2
4,case123_day20_slice_0002,small_bowel,1 1 5 2


In [4]:
missing_ids = [i for i in df_train["id"] if i not in set(df_ssub["id"])]
print(f"missing ids count: {len(missing_ids)}")



missing ids count: 95088


In [5]:
PATTERN = ("case*", "case*_day*", "scans", "*.png")
if df_ssub.empty:
    print("using TRAIN dataset?")
    all_imgs = glob.glob(os.path.join(DATASET_FOLDER, "train", *PATTERN))
else:
    print("using TEST dataset!")
    all_imgs = glob.glob(os.path.join(DATASET_FOLDER, "test", *PATTERN))

print(f"total images found: {len(all_imgs)}")
labels = sorted(df_train["class"].unique())
print(f"unique classes: {labels}")




using TEST dataset!
total images found: 6800
unique classes: ['large_bowel', 'small_bowel', 'stomach']


In [6]:
def extract_details(ip):
    img = os.path.basename(ip)
    im_name, _ = os.path.splitext(img)
    folders = ip.split(os.path.sep)
    case_day = folders[-3].split("_")
    case = case_day[0].replace("case", "")
    day = case_day[1].replace("day", "")
    slice_id = im_name.split("_")[1]
    img_path = img.replace(DATASET_FOLDER + "/", "")
    return {
        "id": f"case{case}_day{day}_slice_{slice_id}",
        "Case": int(case),
        "Day": int(day),
        "Slice": slice_id,
        "image": img,
        "image_path": img_path,
    }


if all_imgs:
    display(pd.DataFrame([extract_details(all_imgs[0])]))



,id,Case,Day,Slice,image,image_path
0,case89_day21_slice_0024,89,21,0024,slice_0024_276_276_1.63_1.63.png,slice_0024_276_276_1.63_1.63.png


In [7]:
df_details = pd.DataFrame(
    [extract_details(ip) for ip in tqdm(all_imgs, desc="Extracting details")]
)
print(f"details dataframe size: {len(df_details)}")
display(df_details.head(3))



Extracting details:   0%|          | 0/6800 [00:00<?, ?it/s]

details dataframe size: 6800


,id,Case,Day,Slice,image,image_path
0,case89_day21_slice_0024,89,21,0024,slice_0024_276_276_1.63_1.63.png,slice_0024_276_276_1.63_1.63.png
1,case89_day21_slice_0013,89,21,0013,slice_0013_276_276_1.63_1.63.png,slice_0013_276_276_1.63_1.63.png
2,case89_day21_slice_0015,89,21,0015,slice_0015_276_276_1.63_1.63.png,slice_0015_276_276_1.63_1.63.png


In [8]:
# Return a tiny constant mask ("1 1") for a small deterministic subset
# (every 20‑th row). All other rows receive an empty mask.
TINY_RLE = "1 1"


def get_prediction(row):
    # row.name holds the integer index when using DataFrame.apply
    if isinstance(row.name, (int, np.integer)) and row.name % 20 == 0:
        return TINY_RLE
    return ""  # empty mask for the majority of rows


df_ssub["predicted"] = df_ssub.apply(get_prediction, axis=1)
df_sub = df_ssub.copy()
display(df_sub.head())



NameError: name 'np' is not defined

In [9]:
submission_path = "submission.csv"
df_sub[["id", "class", "predicted"]].to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

preview = pd.read_csv(submission_path).head()
display(preview)

NameError: name 'df_sub' is not defined